In [9]:
import pandas as pd
from glob import glob
import os
import matplotlib.pyplot as plt
from scipy import ndimage
import skimage as sk
import numpy as np
from PIL import Image

In [4]:
df_key = pd.read_csv('2026.3.4.AnnotationlevelKey.csv',dtype=str,index_col=0)

In [10]:
animal = 'SD185'
gland = 'Right'
df = df_key[df_key['AnimalID']==animal]
df = df[df['Gland.side']==gland]

In [11]:
df.head(n=10)

,Image,Genotype,Timepoint,AnimalID,Gland.side,RegionalNumber,Tumor.Animal,Tissue.ID,MappingID,Max.Region,MapBase
924,194629,WT,IV6,SD185,Right,1,N,Tissue1,1,2,Average 2
925,194629,WT,IV6,SD185,Right,2,N,Tissue2,2,2,Average 2


In [36]:
def open_img(
        img_path: str
    ):
    arr = np.array(Image.open(img_path).convert('L'))
    return arr

def get_tissue_bbox(
        tissue_arr,
        name,
        tissue_id,
        data_path
    ):
    # mask = ndimage.binary_fill_holes(mask)
    # labels = sk.measure.label(mask)
    props = sk.measure.regionprops(tissue_arr)
    largest_obj = max(props, key=lambda p: p.area)
    largest_obj_label = largest_obj.label
    minc = min(props, key=lambda p: p.bbox[0])
    minr = min(props, key=lambda p: p.bbox[1])
    maxc = max(props, key=lambda p: p.bbox[2])
    maxr = max(props, key=lambda p: p.bbox[3])
    img_bbox = tissue_arr[minc.bbox[0]-10:maxc.bbox[2]+10,minr.bbox[1]-10:maxr.bbox[3]+10]
    cropped_mask = (img_bbox==largest_obj_label)*img_bbox
    cropped_mask = ndimage.binary_fill_holes(cropped_mask)
    cropped_mask = sk.img_as_ubyte(cropped_mask)
    # save_path_img = os.path.join(data_path,tissue_id,'cropped_image')
    # save_path_mask = os.path.join(data_path,tissue_id,'cropped_mask')
    # os.makedirs(save_path_img,exist_ok=True)
    # os.makedirs(save_path_mask,exist_ok=True)
    # sk.io.imsave(os.path.join(save_path_img,'cropped_'+name+'.png'),img_bbox,check_contrast=False)
    # sk.io.imsave(os.path.join(save_path_mask,'cropped_mask_'+name+'.png'),cropped_mask,check_contrast=False)
    return img_bbox, cropped_mask

def get_map_region(
        map_arr,
        map_id,
        grey_value):
    #grey_value = grey_value_df.loc[grey_value_df['Mapping_ID'] == map_id, 'Map_Grey_value'].values[0] 
    map_region = (map_arr == grey_value).astype(np.float32)
    map_region = ndimage.binary_fill_holes(map_region)
    labels = sk.measure.label(map_region)
    props = sk.measure.regionprops(labels)
    largest_obj = max(props, key=lambda p: p.area)
    largest_obj_label = largest_obj.label
    masked_map_region = (labels==largest_obj_label)*map_region
    if not map_region.any():
        raise ValueError(f"Map region for ID '{map_id}' (grey value {grey_value}) is empty check map image and grey value key.")
    return masked_map_region

In [37]:
def add_padding(
            img_bbox,
            cropped_mask,
            name,
            tissue_id,
            data_path):
        """ 
        reshape the cropped images to a square based on the largest dim using padding
        saves new arrays with added padding of a constant value: 255 for grey scale image, 0 for masks
        returns save path for later use
        """
        max_dim = max(img_bbox.shape)
        padding_y = (max_dim - img_bbox.shape[0]) // 2
        padding_x = (max_dim - img_bbox.shape[1]) // 2
        pad_img_bbox = np.pad(img_bbox,((padding_y+50,padding_y+50),(padding_x+50,padding_x+50)),mode='constant',constant_values=0)
        pad_mask_bbox = np.pad(cropped_mask,((padding_y+50,padding_y+50),(padding_x+50,padding_x+50)),mode='constant',constant_values=0)
        # save_path_img = os.path.join(data_path,tissue_id,'padded_cropped_image')
        # save_path_mask = os.path.join(data_path,tissue_id,'padded_cropped_mask')
        # os.makedirs(save_path_img,exist_ok=True)
        # os.makedirs(save_path_mask,exist_ok=True)
        # sk.io.imsave(os.path.join(save_path_img,'padded_cropped_'+name+'.png'),pad_img_bbox,check_contrast=False)
        # sk.io.imsave(os.path.join(save_path_mask,'padded_cropped_mask_'+name+'.png'),pad_mask_bbox,check_contrast=False)
        return pad_img_bbox,pad_mask_bbox

In [10]:
tissue = open_img(img_path=r"\\pn.vai.org\projects\steensma\vari-core-generated-data\Involution_Atlas_collab\KG_Registration_Pipeline\Masks_For_Reg_Angle\Tissue2\192281.svs.png")
map = open_img(img_path=r"\\pn.vai.org\projects\steensma\vari-core-generated-data\Involution_Atlas_collab\KG_Registration_Pipeline\GrayscaleMaps\resized_maps\SD183_Left.png")

In [39]:
img_bbox, cropped_mask = get_tissue_bbox(tissue,name=None,tissue_id=None,data_path=None)

In [33]:
map_region = get_map_region(map,map_id=None,grey_value=128)

In [41]:
pad_img, pad_mask = add_padding(img_bbox,cropped_mask,name=None,tissue_id=None,data_path=None)

In [45]:
cropped_mask.dtype

dtype('uint8')

In [23]:
import napari

In [47]:
viewer = napari.view_image(pad_mask)

E:\Temp\ipykernel_43964\398061802.py:1: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(pad_mask)
